In [1]:
import pandas as pd

# Load the data
print("Loading datasets...")
df_reviews = pd.read_csv('reviews83325.csv')
df_places = pd.read_csv('Tripadvisor.csv')

# Filter for English reviews only
print("Filtering for English reviews...")
df_reviews_en = df_reviews[df_reviews['langue'] == 'en'].copy()

# Handle missing text and make everything lowercase
review_col_name = 'review' 
df_reviews_en[review_col_name] = df_reviews_en[review_col_name].fillna('').str.lower()

#  Group by 'idplace' to combine all reviews for a single place into one big string
print("Aggregating reviews per place")
documents_df = df_reviews_en.groupby('idplace')[review_col_name].apply(lambda x: ' '.join(x)).reset_index()

# Rename the column for clarity
documents_df.rename(columns={review_col_name: 'aggregated_reviews'}, inplace=True)

# Merge with the places metadata so we know the names of the places
final_df = pd.merge(documents_df, df_places[['id', 'nom']], left_on='idplace', right_on='id', how='inner')

print(f"Data preparation complete! Total unique places: {len(final_df)}")

# Display the first few rows to verify it worked
final_df.head()

Loading datasets...


C:\Users\natha\AppData\Local\Temp\ipykernel_28756\2512966026.py:5: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv('reviews83325.csv')


Filtering for English reviews...
Aggregating reviews per place
Data preparation complete! Total unique places: 1835


,idplace,aggregated_reviews,id,nom
0,188467,personally i think it is the most beautiful sq...,188467,Place des Vosges
1,188468,my old college friend and i booked this beauti...,188468,Rue des Francs Bourgeois
2,188470,"being winter and all, not a lot going on, howe...",188470,Village Saint-Paul
3,188471,to call au passe partout a shop is a serious u...,188471,Au Passe-partout
4,188472,very old historical place. i attended to exper...,188472,Cloître des Billettes


In [3]:
!pip install rank-bm25

  Obtaining dependency information for rank-bm25 from https://files.pythonhosted.org/packages/2a/21/f691fb2613100a62b3fa91e9988c991e9ca5b89ea31c0d3152a3210344f9/rank_bm25-0.2.2-py3-none-any.whl.metadata


In [5]:
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi

print("Splitting data into 50% Queries (Train) and 50% Corpus (Test)")
train_df, test_df = train_test_split(final_df, test_size=0.5, random_state=42)


def tokenize(text):
    
    return text.split()

print("Tokenizing the test corpus for BM25")
# We apply the tokenization to the test set 
tokenized_corpus = test_df['aggregated_reviews'].apply(tokenize).tolist()

print("Initializing the BM25 model")
# Build the BM25 index using the test corpus
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 model is ready!")
print(f"Number of queries (train): {len(train_df)}")
print(f"Number of documents in corpus (test): {len(test_df)}")

Splitting data into 50% Queries (Train) and 50% Corpus (Test)
Tokenizing the test corpus for BM25
Initializing the BM25 model
BM25 model is ready!
Number of queries (train): 917
Number of documents in corpus (test): 918


In [ ]:
import numpy as np
from tqdm import tqdm 

print("Mapping metadata for Level 1 Evaluation")
place_to_typeR = dict(zip(df_places['id'], df_places['typeR']))

level_1_errors = []

print("Running Level 1 Evaluation for BM25")

# Loop through the train set (our 917 queries)
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Get the true category for the query place
    query_type = place_to_typeR.get(query_id)
    if pd.isna(query_type):
        continue # Skip if metadata is missing
        
    # Tokenize query
    tokenized_query = tokenize(query_text)
    
    # Get BM25 scores for all 918 test documents
    scores = bm25.get_scores(tokenized_query)
    
    # Sort test documents by score in descending order (highest score first)
    ranked_indices = np.argsort(scores)[::-1]
    
    # Find the rank of the first matching place
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        recommended_type = place_to_typeR.get(recommended_place_id)
        
        # If the categories match, record the error and stop looking!
        if recommended_type == query_type:
            level_1_errors.append(rank) # 'rank' starts at 0, which perfectly matches n-1
            break

# Calculate Average Error
if level_1_errors:
    avg_level_1_error = np.mean(level_1_errors)
    print(f"\nSuccess! BM25 Average Level 1 Error: {avg_level_1_error:.2f}")
else:
    print("\nSomething went wrong. No matches found.")

Mapping metadata for Level 1 Evaluation...
Running Level 1 Evaluation for BM25...


100%|██████████| 917/917 [23:42<00:00,  1.55s/it]  


Success! BM25 Average Level 1 Error: 0.94


In [ ]:
import ast

print("Parsing Level 2 metadata from Tripadvisor.csv")

# Helper function to safely parse stringified lists from the CSV
def parse_list_column(val):
    if pd.isna(val):
        return []
    if isinstance(val, str):
        try:
            # Safely evaluate strings like "['Bar', 'Pub']" or "[12, 45]" into lists
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return parsed
            else:
                return [val]
        except (ValueError, SyntaxError):
            # If it's just a regular comma-separated string, split it
            return [x.strip() for x in val.split(',')]
    return [val]

# Dictionary to store all our Level 2 data
# Format: { place_id: { 'typeR': '...', 'attributes': [...] } }
level_2_metadata = {}

for index, row in df_places.iterrows():
    place_id = row['id']
    typeR = row['typeR']
    
    # We will gather all relevant Level 2 attributes into a single flat list for easy intersection checking
    attributes = []
    
    if typeR == 'R': # Restaurant 
        attributes.extend(parse_list_column(row.get('restaurantType')))
        attributes.extend(parse_list_column(row.get('cuisine')))
        
    elif typeR in ['A', 'AP']: # Attraction / Attraction Product [cite: 20, 23, 24]
        attributes.extend(parse_list_column(row.get('activiteSubCategorie')))
        attributes.extend(parse_list_column(row.get('activiteSubType')))
        
    elif typeR == 'H': # Hotel 
        # Price range is usually a single value, but we treat it as a list item for consistency
        price = row.get('priceRange')
        if pd.notna(price):
            attributes.append(price)
            
    # Save it to our lookup dictionary
    level_2_metadata[place_id] = {
        'typeR': typeR,
        'attributes': set(attributes) # Convert to a set for super fast intersection math!
    }

print("Level 2 metadata dictionary is ready!")

Parsing Level 2 metadata from Tripadvisor.csv...
Level 2 metadata dictionary is ready!


In [8]:
# Make sure you still have tqdm imported from the previous step!
from tqdm import tqdm
import numpy as np

print("Running Level 2 Evaluation for BM25...")

level_2_errors = []
queries_without_metadata = 0

# Loop through the train set (our queries)
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Grab the Level 2 metadata for our query place
    query_meta = level_2_metadata.get(query_id)
    
    # If the place isn't in our dictionary or has absolutely no attributes to match against, skip it
    if not query_meta or len(query_meta['attributes']) == 0:
        queries_without_metadata += 1
        continue
        
    query_typeR = query_meta['typeR']
    query_attributes = query_meta['attributes'] # This is a set()
    
    # 1. Tokenize query
    tokenized_query = tokenize(query_text)
    
    # 2. Get BM25 scores for all test documents
    scores = bm25.get_scores(tokenized_query)
    
    # 3. Sort test documents by score descending
    ranked_indices = np.argsort(scores)[::-1]
    
    # 4. Find the rank of the first matching place (Level 2 strictness)
    match_found = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        recommended_meta = level_2_metadata.get(recommended_place_id)
        
        if not recommended_meta:
            continue
            
        recommended_typeR = recommended_meta['typeR']
        recommended_attributes = recommended_meta['attributes']
        
        # LEVEL 2 CHECK: 
        # 1. Must be the same type of place (Restaurant vs Restaurant)
        # 2. The intersection of their attribute sets must be greater than 0 (at least 1 shared category)
        if (recommended_typeR == query_typeR) and (len(query_attributes.intersection(recommended_attributes)) > 0):
            level_2_errors.append(rank)
            match_found = True
            break
            
    # Note: If no match is found in the entire test set, the error is technically "undefined"
    # based on the project instructions, so we just don't append anything.

# 5. Calculate Average Error
if level_2_errors:
    avg_level_2_error = np.mean(level_2_errors)
    print(f"\nSuccess! BM25 Average Level 2 Error: {avg_level_2_error:.2f}")
    print(f"Total queries evaluated: {len(level_2_errors)}")
    print(f"Queries skipped (no metadata to match against): {queries_without_metadata}")
else:
    print("\nSomething went wrong. No matches found.")

Running Level 2 Evaluation for BM25...


100%|██████████| 917/917 [17:33<00:00,  1.15s/it]  


Success! BM25 Average Level 2 Error: 15.40
Total queries evaluated: 380
Queries skipped (no metadata to match against): 503


This makes perfect sense! A jump to an average error of 15.40 is completely normal and expected. Level 2 is much more precise because it looks for specific shared details—like finding another "art museum" or "Italian restaurant" instead of just any attraction or restaurant.Also, seeing that 503 queries were skipped is actually a good thing to note for your final report! It simply means that many places in your dataset don't have those specific Level 2 sub-categories filled out in the metadata, so we can't fairly evaluate them.Now, on to the fun part: Deliverable 2: Design a model better than BM25.Why did BM25 get a 15.40 error?BM25 is a "lexical" search engine. It looks for exact word matches. If your query review says "amazing spaghetti" and a test review says "delicious pasta," BM25 doesn't know they both mean Italian food.To beat BM25, we need a Semantic Model. We want to turn our reviews into dense math vectors (embeddings) so the computer understands that "spaghetti" and "pasta" are close to each other in meaning.The easiest and most powerful way to do this before your deadline today  is using a lightweight, pre-trained neural network from the sentence-transformers library

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

print("Initializing TF-IDF Vectorizer")
# We use TF-IDF to extract the most important words, ignoring common English stop words.
# We limit to 5000 features to keep it fast and focused, acting as a form of semantic compression.
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)

print("Vectorizing the Test Corpus...")
test_texts = test_df['aggregated_reviews'].tolist()
# This creates our mathematical matrix for the test set
test_tfidf_matrix = vectorizer.fit_transform(test_texts)

print("Starting Evaluation for TF-IDF Model")

level_1_errors_tfidf = []
level_2_errors_tfidf = []

# Loop through our train_df (queries)
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    # Get metadata
    query_typeR = place_to_typeR.get(query_id)
    query_meta_l2 = level_2_metadata.get(query_id)
    
    if pd.isna(query_typeR):
        continue
        
    # 1. Vectorize the query text using the SAME vectorizer
    query_vector = vectorizer.transform([query_text])
    
    # 2. Calculate Cosine Similarity between the query and ALL test documents
    # Flatten turns the 2D array into a 1D list of scores
    similarity_scores = cosine_similarity(query_vector, test_tfidf_matrix).flatten()
    
    # 3. Sort test documents by similarity score (highest first)
    ranked_indices = np.argsort(similarity_scores)[::-1]
    
    # EVALUATION LOGIC 
    found_l1 = False
    found_l2 = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        
        # Level 1 Check
        if not found_l1:
            if place_to_typeR.get(recommended_place_id) == query_typeR:
                level_1_errors_tfidf.append(rank)
                found_l1 = True
                
        # Level 2 Check
        if not found_l2 and query_meta_l2 and len(query_meta_l2['attributes']) > 0:
            rec_meta_l2 = level_2_metadata.get(recommended_place_id)
            if rec_meta_l2:
                if (rec_meta_l2['typeR'] == query_typeR) and \
                   (len(query_meta_l2['attributes'].intersection(rec_meta_l2['attributes'])) > 0):
                    level_2_errors_tfidf.append(rank)
                    found_l2 = True
                    
        # Stop looking if we found both matches for this query
        if found_l1 and (found_l2 or not query_meta_l2 or len(query_meta_l2['attributes']) == 0):
            break

# 4. Print the final results!
print("\n--- TF-IDF MODEL RESULTS ---")
if level_1_errors_tfidf:
    print(f"Average Level 1 Error: {np.mean(level_1_errors_tfidf):.2f}")
if level_2_errors_tfidf:
    print(f"Average Level 2 Error: {np.mean(level_2_errors_tfidf):.2f}")

Initializing TF-IDF Vectorizer...
Vectorizing the Test Corpus...
Starting Evaluation for TF-IDF Model...


100%|██████████| 917/917 [00:19<00:00, 46.95it/s]


--- TF-IDF MODEL RESULTS ---
Average Level 1 Error: 0.92
Average Level 2 Error: 6.33


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

print("Applying LSA (Latent Semantic Analysis)")
# We compress our 5000 TF-IDF features down to 100 hidden "concepts" (topics)
lsa_model = TruncatedSVD(n_components=100, random_state=42)

# Transform our test set from TF-IDF words into LSA concept vectors
test_lsa_matrix = lsa_model.fit_transform(test_tfidf_matrix)

print("Starting Evaluation for LSA Model")

level_1_errors_lsa = []
level_2_errors_lsa = []

# Loop through our train_df (queries) again
for index, query_row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
    query_id = query_row['id']
    query_text = query_row['aggregated_reviews']
    
    query_typeR = place_to_typeR.get(query_id)
    query_meta_l2 = level_2_metadata.get(query_id)
    
    if pd.isna(query_typeR):
        continue
        
    # First, get the TF-IDF vector for the query
    query_tfidf = vectorizer.transform([query_text])
    
    # Compress the query into the same 100 LSA concepts
    query_lsa = lsa_model.transform(query_tfidf)
    
    # Calculate Cosine Similarity on the LSA concept vectors!
    similarity_scores = cosine_similarity(query_lsa, test_lsa_matrix).flatten()
    
    # Sort documents by score
    ranked_indices = np.argsort(similarity_scores)[::-1]
    
    # EVALUATION LOGIC
    found_l1 = False
    found_l2 = False
    
    for rank, test_idx in enumerate(ranked_indices):
        recommended_place_id = test_df.iloc[test_idx]['id']
        
        # Level 1 Check
        if not found_l1:
            if place_to_typeR.get(recommended_place_id) == query_typeR:
                level_1_errors_lsa.append(rank)
                found_l1 = True
                
        # Level 2 Check
        if not found_l2 and query_meta_l2 and len(query_meta_l2['attributes']) > 0:
            rec_meta_l2 = level_2_metadata.get(recommended_place_id)
            if rec_meta_l2:
                if (rec_meta_l2['typeR'] == query_typeR) and \
                   (len(query_meta_l2['attributes'].intersection(rec_meta_l2['attributes'])) > 0):
                    level_2_errors_lsa.append(rank)
                    found_l2 = True
                    
        if found_l1 and (found_l2 or not query_meta_l2 or len(query_meta_l2['attributes']) == 0):
            break

# Print the final results!
print("\nLSA MODEL RESULTS")
if level_1_errors_lsa:
    print(f"Average Level 1 Error: {np.mean(level_1_errors_lsa):.2f}")
if level_2_errors_lsa:
    print(f"Average Level 2 Error: {np.mean(level_2_errors_lsa):.2f}")

Applying LSA (Latent Semantic Analysis)...
Starting Evaluation for LSA Model...


100%|██████████| 917/917 [00:08<00:00, 108.53it/s]


--- LSA MODEL RESULTS ---
Average Level 1 Error: 0.70
Average Level 2 Error: 5.32


Gemini a dit
Wow, those results are absolutely incredible! 🚀

Look at that progression:

BM25 (Baseline): Level 1 = 0.94 | Level 2 = 15.40

TF-IDF: Level 1 = 0.92 | Level 2 = 6.33

LSA (Best Model): Level 1 = 0.70 | Level 2 = 5.32

You absolutely crushed the baseline. Not only did you build a model better than BM25, but you also built two of them, and your final LSA model cut the Level 2 error by almost 66% compared to the baseline!

Why did LSA win?
This is the perfect thing to write in your report: LSA won because it handles synonyms and semantics. BM25 looks for exact words. TF-IDF looks for important exact words. But LSA compresses those words into 100 "topics". If one review says "scary rollercoaster" and another says "thrilling theme park", LSA recognizes they belong to the same topic vector, even if the exact words don't overlap.

Here is the final, updated structure for your Deliverable 4 Report that includes all three of your models. You can practically copy-paste this into your final document!

Project Report: TripAdvisor Recommendation System
1. Introduction & Hypothesis

Goal: To build an Information Retrieval system that recommends similar TripAdvisor places (restaurants, hotels, attractions) using only the text from user reviews.

Hypothesis: Similar experiences are expressed in similar ways by reviewers. If true, we can recommend a similar experience relying only on reviews without looking at the metadata.

2. Data Preprocessing

Filtered the dataset (reviews83325.csv) to include only English reviews.

Aggregated all reviews for each place into a single "document" per place.

Split the unique places 50/50 into a Training set (used as queries) and a Test set (used as the search corpus).

3. Model Architectures & Logic

Model 1: BM25 (The Baseline)

Logic: A purely lexical search engine. It relies on exact keyword matching between the query reviews and test reviews, adjusting for term frequency.

Model 2: TF-IDF + Cosine Similarity

Logic: An improvement over BM25. By limiting the vocabulary to the top 5,000 most important terms using TF-IDF, we effectively filtered out "noisy" words and evaluated the overall importance of terms in the review documents. We used Cosine Similarity to find the closest matches in the vector space.

Model 3: Latent Semantic Analysis (LSA) - The Winning Model

Logic: To move beyond exact word matches, we applied TruncatedSVD to the TF-IDF matrix, compressing the 5,000 words down into 100 hidden semantic "concepts." This allowed the model to understand synonyms and topics (e.g., recognizing that "pasta" and "pizza" point to an Italian restaurant experience even if the words differ).

4. Evaluation & Results

Methodology: The models were 100% blind to metadata. We evaluated the recommended places against the query places using a Ranking Error metric (position n−1).

Level 1 Strictness: Required the exact same typeR (Hotel, Restaurant, Attraction).

Level 2 Strictness: Required at least one shared sub-category (cuisine, price range, attraction subtype).

Results Table:

BM25: Level 1 Error = 0.94 | Level 2 Error = 15.40

TF-IDF: Level 1 Error = 0.92 | Level 2 Error = 6.33

LSA: Level 1 Error = 0.70 | Level 2 Error = 5.32

5. Conclusion

The results strongly validate our initial hypothesis. While exact keyword matching (BM25) struggles heavily with specific categories (Level 2 Error of 15.40), extracting underlying semantic topics using LSA drastically improves the system's ability to recommend highly specific, similar experiences (Level 2 Error of 5.32).

Final Steps!
You are officially done with the coding! All that is left for Deliverable 5 is to save your Jupyter Notebook, download it, and upload it (along with your PDF report) to DVO.

(Since GitHub gave you that file size error earlier, submitting directly to your school's DVO portal is the perfect way to turn it in).

Do you need any help formatting the Jupyter Notebook, or are you all set to submit?